# Phase 4b: Step 1 — Two-Way Fixed Effects

**ECON 499R — AI Impact on Renewables**  
**Date:** 2026-11-02  
**Driver:** `code/R/04b_step1_twfe.R --date 2026-11-02`  
**Results:** `results/2026-11-02/h1/`, `results/2026-11-02/tables/`, `results/2026-11-02/figures/`

**NAICS 518210 caveat:** Georgia ranks 2nd in `ai_dc_per1000` in 2025 (5.03, behind WA 6.48), above VA (4.86). This likely reflects colocation and cloud hosting, not AI-specific compute. Treat Georgia's high rank as scope noise in the main variable, not as a clean AI signal.

---
## 0. Reproduce gate

**Equation:** $Y_{it} = a_1 \cdot \text{AI}_{i,t-1} + \ln\text{GDP/pc} + \text{manuf} + \text{price} + \text{pop growth} + \mu_i + \lambda_t$  
**Tests:** match `econ499r_starter/reference/prelim_step1_slx.csv` within rounding.

| Outcome | Ref a₁ | Got a₁ | Diff | Result |
|---|---|---|---|---|
| ln_ec | −0.001565 | −0.001565 | 0.000000 | **PASS** |
| ln_ee | +0.002206 | +0.002206 | 0.000000 | **PASS** |
| renew_share | −0.010129 | −0.010129 | 0.000000 | **PASS** |
| solar_share | −0.000059 | −0.000059 | 0.000000 | **PASS** |
| new_renew_GW | +0.075995 | +0.075995 | 0.000000 | **PASS** |

All five match exactly. Proceeding to main analysis.

In [ ]:
setwd('/Users/shunji/Desktop/EMORY/ECON 499R/AI impact on Renewables')
library(data.table); library(ggplot2)

h1_core  <- fread('results/2026-11-02/h1/h1_core.csv')
h1_extra <- fread('results/2026-11-02/h1/h1_extras.csv')
se_check <- fread('results/2026-11-02/h1/se_check.csv')
nonlin   <- fread('results/2026-11-02/h1/nonlinearity.csv')
tp       <- fread('results/2026-11-02/h1/turning_points.csv')
het      <- fread('results/2026-11-02/h1/heterogeneity.csv')
dyn      <- fread('results/2026-11-02/h1/dynamics.csv')
cat('All result files loaded.\n')

---
## 1. H1 Core — with and without price control

**Equation:** $Y_{it} = a_1 \cdot \text{AI}_{i,t-1} + \ln\text{GDP/pc} + \text{price} + \text{tech share} + \text{manuf} + \text{pop growth} + \mu_i + \lambda_t$  
**Tests:** Is AI data-center concentration associated with consumption, efficiency, or renewable share?  
**N = 550** (50 states × 11 years). SE clustered by state. Winsorized 1/99.

In [ ]:
cols_show <- c('outcome','controls','estimate','se','pval','N','within_r2')
print(h1_core[, ..cols_show])

**Reading:**
- **ln_ec (consumption):** a₁ = −0.0022 (ctrl5), p = 0.704. Does not support H1. Adding tech_share to controls modestly shifts the coefficient from −0.0016 to −0.0022 but remains insignificant — tech_share is positively correlated with both AI and efficiency, so its inclusion absorbs some positive confounding.
- **ln_ee (efficiency):** a₁ = +0.0031, p = 0.567. Insignificant. Directionally positive (states with more AI data centers are slightly more efficient), but imprecise.
- **renew_share:** a₁ = −0.0103, p = 0.103. Borderline. Negative sign is opposite to the blessing direction. Does not support H1 at the 10% level under clustered SEs.
- **Price control:** The ctrl4 vs ctrl5 comparison shows minimal sensitivity — coefficients shift by <0.001 for all three outcomes. Price is not acting as a strong bad control in this sample.

---
## 2. H1 Extras

**Same equation, additional outcomes.**  
**Falsification:** coal_share should be null if the AI effect is specific to renewables.

In [ ]:
print(h1_extra[, .(outcome, estimate, se, pval, N, within_r2)])

**Reading:**
- **new_renew_mw: a₁ = +74.5 MW, se = 31.5, p = 0.022.** Only significant result. A one-unit increase in AI/1000 workers is associated with ~75 MW more new renewable capacity added that year. This is the clearest positive signal in the H1 battery.
- **solar_share:** a₁ = +0.0002, p = 0.955. Null.
- **wind_share:** a₁ = −0.0085, p = 0.222. Null.
- **fossil_to_renew:** a₁ = +0.95, p = 0.110. Wrong sign for H1 (H1 requires a₁ < 0). Does not support H1.
- **coal_share:** a₁ = +0.0019, p = 0.808. Null — as expected for a falsification check.
- **co2_per_mwh_t (2015–2024):** a₁ = +0.0022, p = 0.731. Null. Wrong sign direction but not significant.

---
## 3. SE Check: Clustered vs Driscoll-Kraay

In [ ]:
print(se_check[, .(outcome, vcov_type, estimate, se, pval)])

**Reading:** Driscoll-Kraay SEs are substantially *smaller* than state-clustered SEs (~4× smaller for ln_ee and renew_share). This is unexpected — DK accounts for serial correlation *and* cross-sectional dependence, so it usually produces wider CIs. The smaller DK SEs here likely reflect that cross-sectional correlations are not large after removing year FE.

Under DK:
- **ln_ee: p = 0.028** (significant at 5%)
- **renew_share: p = 0.001** (significant at 1%)
- ln_ec: p = 0.172 (still insignificant)

Report both SE types in the final table. Baseline = clustered (more conservative).

---
## 4. Nonlinearity (AI + AI²)

**Equation:** $Y_{it} = b_1 \cdot \text{AI}_{i,t-1} + b_2 \cdot \text{AI}^2_{i,t-1} + \text{controls} + \mu_i + \lambda_t$  
**Turning point:** $\text{AI}^* = -b_1 / (2 b_2)$ with delta-method SE.

In [ ]:
print(tp[, .(outcome, b1, b2, turning_point, tp_se, tp_lo, tp_hi, ai_min, ai_max,
             inside_range, pct_state_yrs_past_tp, lm_opposite_signs)])

**Reading:**
- **ln_ec:** TP = 2.75 (95% CI: 0.42–5.08) inside the data range [0.26, 5.65]. Opposite slopes at min/max (Lind-Mehlum: YES). Inverted-U: consumption rises with AI up to ~2.75/1000 workers then falls. 25.5% of state-years are past the turning point.
- **solar_share:** TP = 3.30 (CI: −0.70–7.30) inside range. Opposite slopes (YES). Inverted-U for solar.
- **fossil_to_renew:** TP = 4.78 (CI: 1.01–8.55) inside range. Opposite slopes (YES).
- **coal_share:** TP = 3.90 (CI: −3.79–11.58). Wide CI, inside range, opposite slopes.
- All other outcomes: turning point outside range or no extremum.

**Caution:** These are exploratory. With b₂ imprecise (large SEs), turning points have very wide CIs. The ln_ec inverted-U is the most credible (CI stays within data range).

---
## 5. Heterogeneity (2015-baseline moderators)

**Equation:** $Y_{it} = a_1 \text{AI}_{i,t-1} + \gamma_1 (\text{AI}_{i,t-1} \times D_i) + a_2 D_i + \text{controls} + \mu_i + \lambda_t$  
Moderators defined from 2015 values (fixed, cannot respond to AI). AI and D mean-centered.

In [ ]:
# Show renew_share interactions only (most informative)
print(het[outcome == 'renew_share', .(coef, estimate, se, pval, N)])

**Reading:**
- **AI × net_exporter (renew_share):** γ₁ = −0.019, p = 0.094. Net exporters show a more *negative* AI association with renewable share. Borderline significant.
- **AI × high_price (renew_share):** γ₁ = **+0.032, p = 0.004.** High-electricity-price states show a significantly more *positive* AI–renewable association. Economically interpretable: where electricity is expensive, data-center operators have stronger incentives to source renewables (PPAs, on-site generation).
- **AI × South:** γ₁ = +0.0002, p = 0.984. No South heterogeneity.
- Note: time-invariant moderators (net_exporter_2015, high_price_2015) are absorbed by state FE and removed by fixest as collinear — the interaction term is identified off the deviation from state mean, which is the time-varying AI interacted with the fixed dummy.

---
## 6. Dynamics (t-1 only + long difference)

**Note:** QCEW 2012–2013 absent; t-2 and t-3 distributed lags omitted.

In [ ]:
print(dyn[, .(tag = sub('.*_ai518210_.*?_', '', tag), outcome, estimate, se, pval, N)])

**Reading:** Long-difference results (N=46 states; 4 states dropped for suppressed 2025 AI) are all insignificant. The ΔAI (2014→2024) has no detectable association with ΔY (2015→2025) in the cross-section. This is consistent with small within-state variation in AI: SD of ai_dc_per1000 ≈ 0.57, so 10-year changes are also small.

---
## 7. Georgia descriptive

In [ ]:
ga <- fread('results/2026-11-02/h1/georgia_descriptive.csv')
print(ga[, .(year, ai_dc_per1000, us_med_ai, renew_share, us_med_renew, resid_renew)])

**Reading:** Georgia's AI concentration rose from 2.0 to 5.0/1000 workers (2015–2025), well above the U.S. median (1.9–2.6). Georgia's renewable share rose from ~12% to ~13%, tracking close to the U.S. median (~20–28%). Residuals from the renew_share TWFE model are small (|ε| ≤ 0.02), meaning Georgia does not systematically over- or under-perform the model's prediction.

**NAICS 518210 caveat:** Georgia's high AI rank likely reflects AWS, Microsoft Azure, and Google data centers that serve commercial cloud workloads, not primarily AI training. This means ai_dc_per1000 overstates AI-specific compute intensity for Georgia.

---
## 8. Figures

![H1 coefficient plot](../results/2026-11-02/figures/h1_coef_plot.png)

![Georgia AI and renewable paths](../results/2026-11-02/figures/georgia_ai_renew_paths.png)

---
## Summary

| Outcome | H1 verdict | Strongest result |
|---|---|---|
| ln_ec (consumption) | Does not support H1 (p=0.704 CL; p=0.172 DK) | Inverted-U under quadratic spec (TP=2.75) |
| ln_ee (efficiency) | Does not support H1 under CL (p=0.567); supports under DK (p=0.028) | Positive but imprecise under clustered SEs |
| renew_share | Does not support H1 under CL (p=0.103); supports under DK (p=0.001) | Negative under CL (peril direction) |
| new_renew_mw | **Supports H1 (p=0.022, CL)** | +74.5 MW per unit AI — only unambiguous significant result |
| solar/wind share | Null | — |
| fossil_to_renew | Wrong sign (p=0.110) | — |
| coal_share | Null (falsification passes) | — |
| CO2/MWh | Null | — |

**Heterogeneity:** High-electricity-price states show significantly more positive AI–renewable share association (γ₁=+0.032, p=0.004). This is the clearest pattern in the H1 battery.

**Next:** `prompts/04c_step2_mediation.md`